Check the model names from Pytorch Computer Vision (Timm) library:

In [1]:
# Hyperparameters
MODEL = "seresnext101_32x4d.gluon_in1k_timm"
DEVICE_ID = 7
BATCH_SIZE = 128
IMAGE_SIZE = 224
EPOCHS = 100
LR = 1e-4
OPTIMIZER = "AdamW"
DATA_DIR = r'/home/c/choton/beemachine/datasets/Others/CUB_200_2011/'
LOG_DIR = f"./{MODEL}_logs/"
SEED = 42

In [2]:
# Pytorch vision package (Timm)
import timm

# List relevant models
timm_name = MODEL[:-5] # Removing _timm in the name
convnext_models = timm.list_models(f'{timm_name}*')
print(convnext_models)

pretrained_models = timm.list_models(f'{timm_name}*', pretrained=True)
print(pretrained_models)

[]
['seresnext101_32x4d.gluon_in1k']


In [3]:
# Pytorch packages
import torch
from torch import nn
# from torch.nn import functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# Other packages
import os
import json
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Seeding for consistant results
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [4]:
# =========================
# 2. LOAD OFFICIAL SPLITS
# =========================

images_df = pd.read_csv(
    os.path.join(DATA_DIR, "images.txt"),
    sep=" ",
    names=["img_id", "filepath"]
)

labels_df = pd.read_csv(
    os.path.join(DATA_DIR, "image_class_labels.txt"),
    sep=" ",
    names=["img_id", "label"]
)

split_df = pd.read_csv(
    os.path.join(DATA_DIR, "train_test_split.txt"),
    sep=" ",
    names=["img_id", "is_train"]
)

# Merge all
df = images_df.merge(labels_df, on="img_id")
df = df.merge(split_df, on="img_id")

# Convert labels to 0-index
df["label"] = df["label"] - 1

# =========================
# 3. CREATE 75:15:10 SPLIT
# =========================

# First separate official test
official_train = df[df["is_train"] == 1]
official_test = df[df["is_train"] == 0]

# Combine all data
full_df = pd.concat([official_train, official_test])

# 75% train, 25% temp
train_df, temp_df = train_test_split(
    full_df,
    test_size=0.25,
    stratify=full_df["label"],
    random_state=SEED
)

# Split temp into val (15%) and test (10%)
val_ratio = 0.15 / (0.15 + 0.10)

val_df, test_df = train_test_split(
    temp_df,
    test_size=(1 - val_ratio),
    stratify=temp_df["label"],
    random_state=SEED
)

print(f"Train: {len(train_df)}, Val:   {len(val_df)}, Test:  {len(test_df)}")

Train: 8841, Val:   1768, Test:  1179


In [5]:
class CUBDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(DATA_DIR, "images", row["filepath"])
        image = Image.open(img_path).convert("RGB")
        label = row["label"]
        if self.transform:
            image = self.transform(image)
        return image, label

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = CUBDataset(train_df, train_transform)
val_dataset = CUBDataset(val_df, val_transform)
test_dataset = CUBDataset(test_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, persistent_workers=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True)

classes = set(labels_df['label'])
num_classes = len(classes)
print(f"Classes: {num_classes} | Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: 200 | Train: 8841 | Val: 1768 | Test: 1179


In [7]:
# Get one batch from the train loader
images, labels = next(iter(train_loader))

print("Batch image tensor shape:", images.shape)
print("Batch label tensor shape:", labels.shape)

Batch image tensor shape: torch.Size([128, 3, 224, 224])
Batch label tensor shape: torch.Size([128])


In [8]:
# Loading the model
device = torch.device(f"cuda:{DEVICE_ID}")
num_classes = len(classes)
model = timm.create_model(timm_name, pretrained=True, num_classes=num_classes)
model.to(device)

# Setting up the criterion, optimizer and scheduler
criterion = nn.CrossEntropyLoss()  #label_smoothing=0.1)
if OPTIMIZER == "SGD":
    optimizer = torch.optim.SGD(
        model.parameters(), 
        lr=LR, 
        momentum=0.9, 
        weight_decay=1e-4)
elif OPTIMIZER == "AdamW":
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LR, 
        weight_decay=1e-4)
else:
    optimizer = torch.optim.RMSprop(
        model.parameters(),
        lr=LR,       # typically around 0.256 / batch_size scaling, you can tune
        alpha=0.9,              # decay (smoothing constant)
        eps=0.001,              # numerical stability
        momentum=0.9,           # as in paper
        weight_decay=1e-5,      # L2 regularization
        centered=False          # same as paper’s setup
    )
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

In [9]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0, 0, 0

    pbar = tqdm(train_loader, desc="[Train]", position=0, leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc="[Val]", position=0, leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [10]:
%%time

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...


[Train]:   0%|          | 0/70 [00:00<?, ?it/s]

 Epoch 1/100 | Train Loss: 3.7729 | Train Acc: 0.3071 | Val Loss: 1.7422 | Val Acc: 0.6160


 Epoch 2/100 | Train Loss: 1.1505 | Train Acc: 0.7464 | Val Loss: 0.9681 | Val Acc: 0.7500


 Epoch 3/100 | Train Loss: 0.5049 | Train Acc: 0.8790 | Val Loss: 0.8504 | Val Acc: 0.7568


...

 Epoch 96/100 | Train Loss: 0.0028 | Train Acc: 1.0000 | Val Loss: 0.7505 | Val Acc: 0.8218


 Epoch 97/100 | Train Loss: 0.0028 | Train Acc: 0.9999 | Val Loss: 0.7601 | Val Acc: 0.8230


 Epoch 98/100 | Train Loss: 0.0029 | Train Acc: 0.9999 | Val Loss: 0.7580 | Val Acc: 0.8235


 Epoch 99/100 | Train Loss: 0.0031 | Train Acc: 0.9999 | Val Loss: 0.7460 | Val Acc: 0.8235


 Epoch 100/100 | Train Loss: 0.0031 | Train Acc: 1.0000 | Val Loss: 0.7457 | Val Acc: 0.8213
Training complete!
CPU times: user 1h 11min 46s, sys: 17min 9s, total: 1h 28min 55s
Wall time: 1h 40min 33s


In [12]:
test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 0.7966 | Test Acc: 0.8151
